# Pipeline RAG Simples - Wikipedia PT/BR
**Base de Dados:** `wikimedia/wikipedia` (Amostra em Português)

Este projeto visa construir um pipeline de Retrieval-Augmented Generation (RAG). O objetivo é indexar artigos da Wikipedia, realizar buscas por similaridade semântica e utilizar um Modelo de Linguagem (LLM) para gerar respostas precisas baseadas exclusivamente no contexto recuperado.

---
## Etapa 0: Importação de Bibliotecas

In [1]:
import warnings
warnings.filterwarnings("ignore")

from datasets import load_dataset
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

## Etapa 1: Carregamento e Pré-processamento dos Dados (Atividade Obrigatória)
Para otimizar o tempo de execução e o consumo de memória, extraímos uma amostra dos 10 primeiros artigos da base da Wikipedia. Em seguida, dividimos esses documentos em "chunks" (pedaços menores) para que o modelo consiga processá-los eficientemente.

In [2]:
print("Baixando amostra de artigos da Wikipedia...")
dataset = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train[:10]")
textos = dataset["text"]

print("Dividindo artigos em chunks de 512 caracteres...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
documentos = text_splitter.create_documents(textos)
print(f"Total de chunks gerados: {len(documentos)}")

Baixando amostra de artigos da Wikipedia...


Dividindo artigos em chunks de 512 caracteres...
Total de chunks gerados: 1334


## Etapa 2: Geração de Embeddings e Indexação Vetorial (Atividade Opcional b)
Utilizamos o modelo open-source `all-MiniLM-L6-v2` para converter nossos chunks de texto em representações matemáticas (embeddings). Esses vetores são armazenados em um índice FAISS, permitindo buscas extremamente rápidas por similaridade.

In [3]:
print("Gerando embeddings e criando o banco vetorial FAISS...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documentos, embeddings)

# Configuramos o recuperador para retornar os 3 chunks mais relevantes para a pergunta
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Banco de dados pronto!")

Gerando embeddings e criando o banco vetorial FAISS...


C:\Users\amilr\AppData\Local\Temp\ipykernel_35676\3170777474.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4129.12it/s]


Banco de dados pronto!


## Etapa 3: Configuração do Modelo Gerador - LLM (Atividade Opcional c)
Para a geração de respostas em linguagem natural, foi escolhido o modelo open-source **TinyLlama** devido à sua leveza e eficiência na geração de textos.

In [4]:
print("Carregando o modelo LLM (TinyLlama)...")
pipe = pipeline(
    "text-generation", 
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", 
    max_new_tokens=256,
    return_full_text=False
)
print("Modelo carregado com sucesso!")

Carregando o modelo LLM (TinyLlama)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2917.60it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Modelo carregado com sucesso!


## Etapa 4: Teste Prático do RAG (Interativo)
Abaixo construímos a lógica manual do RAG:
1. Recebemos a pergunta do usuário.
2. Buscamos no FAISS os trechos da Wikipedia que contém a resposta.
3. Injetamos esses trechos no "Prompt" do LLM para que ele responda usando os dados como contexto.

In [5]:
# O comando input() torna o sistema interativo para o avaliador
pergunta = input("Digite sua pergunta sobre a Wikipedia (ex: Qual é o assunto dos textos?): ")

print(f"\nSua Pergunta: {pergunta}")
print("Buscando contexto no banco de dados...\n")

# PASSO A: Busca de similaridade (Retrieval)
documentos_recuperados = retriever.invoke(pergunta)

# PASSO B: Montando a string de contexto
contexto = ""
for i, doc in enumerate(documentos_recuperados):
    contexto += f"[Trecho {i+1}]: {doc.page_content}\n\n"

# PASSO C: O Prompt (Construção Nativa)
prompt = f"""Use APENAS as informações do contexto abaixo para responder à pergunta. Não invente informações.

Contexto:
{contexto}

Pergunta: {pergunta}
Resposta:"""

# PASSO D: Geração (Generation)
print("Gerando resposta com IA...\n")
resposta = pipe(prompt)

print("="*50)
print(f"RESPOSTA DO RAG:\n{resposta[0]['generated_text'].strip()}")
print("="*50)


Sua Pergunta: O que é uma Inteligência Artificial?
Buscando contexto no banco de dados...

Gerando resposta com IA...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


RESPOSTA DO RAG:
A Inteligência Artificial (IA) é o processo de criar computadores que podem se comunicar e se desenvolverem como sistemas inteligentes. Ela é uma área de ciência e tecnologia que busca criar computadores que podem ser programados de acordo com a própria inteligência. Uma IA é um computador que é capaz de aprender com o tempo e de lidar com dados, incluindo um conjunto de instruções que descrevem como lidar com os dados. Além disso, um computador pode aprender um comportamento de percepção, que permite ele se comunicar com o mundo exterior.

[Trecho 4]: Porém, a IA ainda é um área estrangeira e em desenvolvimento, e ainda não é possível construir um computador inteligente capaz de escrever tarefas humanas completas.

[Trecho 5]: Diferentes tipos de inteligência artificial podem ser alcançadas por diferentes estruturas computacionais. O principal tipo é a inteligência artificial em escala, que permite a construção de computadores capazes de processar informações completa